# Pose Normalization

Per-frame normalization using shoulder center as origin and torso size as scale.

**Input:** `keypoints_combined`  
**Output:** shoulder center = (0, 0), torso size = 1

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IN_DIR      = Path('../data/processed/keypoints_corrected')
OUT_DIR     = Path('../data/processed/keypoints_normalized_corrected')
DOWN_IN_DIR  = Path('../data/processed/keypoints_corrected_down')
DOWN_OUT_DIR = Path('../data/processed/keypoints_normalized_corrected_down')

JOINTS = [
    'nose',
    'left_shoulder', 'right_shoulder',
    'left_elbow',    'right_elbow',
    'left_wrist',    'right_wrist',
    'left_hip',      'right_hip',
    'left_knee',     'right_knee',
    'left_ankle',    'right_ankle',
]

## Normalization function

Frames with missing reference joints (shoulder or hip) are left as NaN.

In [2]:
def normalize_pose(df):
    df = df.copy()

    sc_x = (df['left_shoulder_x'] + df['right_shoulder_x']) / 2
    sc_y = (df['left_shoulder_y'] + df['right_shoulder_y']) / 2

    hc_x = (df['left_hip_x'] + df['right_hip_x']) / 2
    hc_y = (df['left_hip_y'] + df['right_hip_y']) / 2

    torso_size = np.sqrt((sc_x - hc_x) ** 2 + (sc_y - hc_y) ** 2)
    invalid = (torso_size == 0) | torso_size.isna()

    for joint in JOINTS:
        df[f'{joint}_x'] = (df[f'{joint}_x'] - sc_x) / torso_size
        df[f'{joint}_y'] = (df[f'{joint}_y'] - sc_y) / torso_size
        df.loc[invalid, f'{joint}_x'] = np.nan
        df.loc[invalid, f'{joint}_y'] = np.nan

        if f'{joint}_z' in df.columns:
            sc_z = (df['left_shoulder_z'] + df['right_shoulder_z']) / 2
            df[f'{joint}_z'] = (df[f'{joint}_z'] - sc_z) / torso_size
            df.loc[invalid, f'{joint}_z'] = np.nan

    return df

In [3]:
for in_dir, out_dir, label in [
    (IN_DIR, OUT_DIR, 'sit-to-stand'),
    (DOWN_IN_DIR, DOWN_OUT_DIR, 'stand-to-sit'),
]:
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n=== {label} ===')
    for csv_path in sorted(in_dir.glob('*.csv')):
        df      = pd.read_csv(csv_path)
        df_norm = normalize_pose(df)
        df_norm.to_csv(out_dir / csv_path.name, index=False)
        print(f'{csv_path.stem}: {len(df)} frames')


=== sit-to-stand ===
DJI_20250425092743_0028_D: 94 frames
DJI_20250425093100_0030_D: 92 frames


DJI_20250425104507_0045_D: 73 frames
DJI_20250425104804_0047_D: 94 frames
DJI_20250425112502_0059_D: 123 frames


DJI_20250425112749_0061_D: 151 frames


DJI_20250425120835_0074_D: 60 frames


DJI_20250425121226_0076_D: 151 frames
DJI_20250425125202_0091_D: 61 frames


...

DJI_20250425093100_0030_D: 175 frames
DJI_20250425104507_0045_D: 67 frames
DJI_20250425104804_0047_D: 156 frames


DJI_20250425112502_0059_D: 98 frames
DJI_20250425112749_0061_D: 142 frames


DJI_20250425120835_0074_D: 47 frames


DJI_20250425121226_0076_D: 129 frames


DJI_20250425125202_0091_D: 32 frames
DJI_20250425125448_0093_D: 135 frames


## Verify

Shoulder center should be (0, 0) in every normalized frame.

In [4]:
csv_path = sorted(OUT_DIR.glob('*.csv'))[0]
df = pd.read_csv(csv_path).dropna(subset=['left_shoulder_x'])

sc_x = (df['left_shoulder_x'] + df['right_shoulder_x']) / 2
sc_y = (df['left_shoulder_y'] + df['right_shoulder_y']) / 2

print(f'{csv_path.stem}')
print(f'  shoulder center x — mean: {sc_x.mean():.6f}, max abs: {sc_x.abs().max():.6f}')
print(f'  shoulder center y — mean: {sc_y.mean():.6f}, max abs: {sc_y.abs().max():.6f}')

DJI_20250425092743_0028_D
  shoulder center x — mean: -0.000000, max abs: 0.000000
  shoulder center y — mean: -0.000000, max abs: 0.000000


## Step 6: Temporal Stability

Frame-to-frame Euclidean displacement per joint in normalized space.


Lower = smoother tracking.

### Stability per video

In [5]:
rows = []

for csv_path in sorted(OUT_DIR.glob('*.csv')):
    if csv_path.name.startswith('stability'):
        continue
    df = pd.read_csv(csv_path)
    disps = []
    for joint in JOINTS:
        x = df[f'{joint}_x'].values
        y = df[f'{joint}_y'].values
        d = np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2)
        disps.extend(d[~np.isnan(d)])
    rows.append({
        'Video':             csv_path.stem,
        'Mean Displacement': round(np.mean(disps), 4),
        'Std Displacement':  round(np.std(disps), 4),
    })

df_overall = pd.DataFrame(rows)
df_overall.to_csv(OUT_DIR / 'stability_per_video.csv', index=False)
df_overall

,Video,Mean Displacement,Std Displacement
0,DJI_20250425092743_0028_D,0.0103,0.0116
1,DJI_20250425093100_0030_D,0.0104,0.0109
2,DJI_20250425104507_0045_D,0.0117,0.0126
3,DJI_20250425104804_0047_D,0.0094,0.0095
4,DJI_20250425112502_0059_D,0.0106,0.0096
5,DJI_20250425112749_0061_D,0.0066,0.0079
6,DJI_20250425120835_0074_D,0.0131,0.0132
7,DJI_20250425121226_0076_D,0.0058,0.0090
8,DJI_20250425125202_0091_D,0.0135,0.0120
9,DJI_20250425125448_0093_D,0.0086,0.0108


### Per-joint stability

To identifying which joints have higher jitter.

In [6]:
rows_joint = []

for csv_path in sorted(OUT_DIR.glob('*.csv')):
    if csv_path.name.startswith('stability'):
        continue
    df = pd.read_csv(csv_path)
    for joint in JOINTS:
        x = df[f'{joint}_x'].values
        y = df[f'{joint}_y'].values
        d = np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2)
        d = d[~np.isnan(d)]
        if len(d) == 0:
            continue
        rows_joint.append({
            'Video':             csv_path.stem,
            'Joint':             joint,
            'Mean Displacement': round(np.mean(d), 4),
            'Std Displacement':  round(np.std(d), 4),
        })

df_joint = pd.DataFrame(rows_joint)
df_joint.to_csv(OUT_DIR / 'stability_per_joint.csv', index=False)
df_joint

,Video,Joint,Mean Displacement,Std Displacement
0,DJI_20250425092743_0028_D,nose,0.0061,0.0052
1,DJI_20250425092743_0028_D,left_shoulder,0.0035,0.0034
2,DJI_20250425092743_0028_D,right_shoulder,0.0035,0.0034
3,DJI_20250425092743_0028_D,left_elbow,0.0110,0.0104
4,DJI_20250425092743_0028_D,right_elbow,0.0134,0.0085
...,...,...,...,...
125,DJI_20250425125448_0093_D,right_hip,0.0057,0.0052
126,DJI_20250425125448_0093_D,left_knee,0.0109,0.0123
127,DJI_20250425125448_0093_D,right_knee,0.0116,0.0136
128,DJI_20250425125448_0093_D,left_ankle,0.0142,0.0170


## Step 7: Standardization

In [7]:
from sklearn.preprocessing import StandardScaler

STD_DIR      = Path('../data/processed/keypoints_standardized')
DOWN_STD_DIR = Path('../data/processed/keypoints_standardized_down')

COORD_COLS = [f'{joint}_{c}' for joint in JOINTS for c in ['x', 'y']]

for norm_dir, std_dir, label in [
    (OUT_DIR, STD_DIR, 'sit-to-stand'),
    (DOWN_OUT_DIR, DOWN_STD_DIR, 'stand-to-sit'),
]:
    std_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n=== {label} ===')

    all_frames = pd.concat(
        [pd.read_csv(f)[COORD_COLS] for f in sorted(norm_dir.glob('*.csv'))
         if not f.name.startswith('stability')],
        ignore_index=True
    )

    scaler = StandardScaler()
    scaler.fit(all_frames.dropna())

    for csv_path in sorted(norm_dir.glob('*.csv')):
        if csv_path.name.startswith('stability'):
            continue
        df = pd.read_csv(csv_path)
        valid = df[COORD_COLS].notna().all(axis=1)
        if valid.any():
            df.loc[valid, COORD_COLS] = scaler.transform(df.loc[valid, COORD_COLS])
        df.to_csv(std_dir / csv_path.name, index=False)
        print(f'{csv_path.name} -> standardized')


=== sit-to-stand ===


DJI_20250425092743_0028_D.csv -> standardized
DJI_20250425093100_0030_D.csv -> standardized
DJI_20250425104507_0045_D.csv -> standardized
DJI_20250425104804_0047_D.csv -> standardized
DJI_20250425112502_0059_D.csv -> standardized


DJI_20250425112749_0061_D.csv -> standardized
DJI_20250425120835_0074_D.csv -> standardized


DJI_20250425121226_0076_D.csv -> standardized
DJI_20250425125202_0091_D.csv -> standardized
DJI_20250425125448_0093_D.csv -> standardized

=== stand-to-sit ===


DJI_20250425092743_0028_D.csv -> standardized


DJI_20250425093100_0030_D.csv -> standardized
DJI_20250425104507_0045_D.csv -> standardized


DJI_20250425104804_0047_D.csv -> standardized
DJI_20250425112502_0059_D.csv -> standardized
DJI_20250425112749_0061_D.csv -> standardized
DJI_20250425120835_0074_D.csv -> standardized


DJI_20250425121226_0076_D.csv -> standardized
DJI_20250425125202_0091_D.csv -> standardized


DJI_20250425125448_0093_D.csv -> standardized
